# 04 — Zero-shot, one-shot, few-shot

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase02/notebooks/04_prompting_techniques.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objetivo.** Comparar las tres técnicas en una tarea concreta: clasificación de sentimiento. Vas a ver cómo unos pocos ejemplos cambian el comportamiento.


In [ ]:
%pip install --quiet groq


In [ ]:
import os
from groq import Groq

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    assert os.environ.get("GROQ_API_KEY"), "Exportá GROQ_API_KEY."

client = Groq()
MODEL = "qwen/qwen3-32b"

def clasificar(messages, temperature=0.1):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        # reasoning_effort="none",  # descomentar para que Qwen3 no devuelva <think>...</think>
    )
    return resp.choices[0].message.content.strip()


## Reseñas de prueba

Mezclamos casos fáciles y ambiguos para ver dónde fallan las distintas técnicas.


In [ ]:
RESEÑAS = [
    "El servicio fue impecable, vuelvo seguro.",                                 # claramente positiva
    "Pésimo, no me devolvieron la plata.",                                       # claramente negativa
    "Está OK, nada del otro mundo pero cumple.",                                 # neutral
    "El servicio fue lento pero la comida estuvo buena.",                        # ambiguo
    "Llegó a tiempo, eso es lo único bueno que puedo decir.",                    # sarcástico/negativo
    "Hace lo que promete, ni más ni menos.",                                     # neutral
]


## Zero-shot

Solo le decimos qué hacer, sin ejemplos.


In [ ]:
SYSTEM_ZS = "Sos un clasificador de sentimientos. Respondés SOLO con una palabra: POSITIVA, NEUTRAL o NEGATIVA."

print("ZERO-SHOT")
print("-" * 60)
for r in RESEÑAS:
    out = clasificar([
        {"role": "system", "content": SYSTEM_ZS},
        {"role": "user", "content": f"Reseña: {r}"},
    ])
    print(f"  {out:<10} <- {r}")


## Few-shot

Le damos 3 ejemplos resueltos. Observá cómo mejora la consistencia en los casos ambiguos.


In [ ]:
SYSTEM_FS = "Sos un clasificador de sentimientos. Respondés SOLO con: POSITIVA, NEUTRAL o NEGATIVA."

EJEMPLOS = '''Reseña: Excelente atención y precio justo.
Etiqueta: POSITIVA

Reseña: Lo recibí roto y nadie responde.
Etiqueta: NEGATIVA

Reseña: Cumple, pero tarda más que la competencia.
Etiqueta: NEUTRAL

'''

print("FEW-SHOT")
print("-" * 60)
for r in RESEÑAS:
    user = EJEMPLOS + f"Reseña: {r}\nEtiqueta:"
    out = clasificar([
        {"role": "system", "content": SYSTEM_FS},
        {"role": "user", "content": user},
    ])
    print(f"  {out:<10} <- {r}")


## Para experimentar

- Cambiá los ejemplos del few-shot por casos más cercanos a tu dominio. Probablemente mejore más.
- Probá con 1 solo ejemplo (one-shot) y compará con few-shot.
- Pedí al modelo que devuelva además una **explicación** de por qué clasificó así. Esto es CoT, lo vemos en el próximo notebook.
- Probá con `temperature=0.7` en zero-shot — ¿se vuelve más inconsistente?
